In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow
print("\n✅ Cài xong.")


## Tải dataset ký tự từ Roboflow

⭐ **Đừng gõ thẳng API key vào code** nếu định Save Version / public notebook — ai xem code cũng
lấy được key. Dùng **Kaggle Secrets** (Add-ons → Secrets → thêm `ROBOFLOW_API_KEY`) rồi đọc ra như dưới đây.

In [ ]:
from roboflow import Roboflow

# ⭐ Lấy key từ Kaggle Secrets thay vì gõ thẳng vào code (an toàn hơn khi share/public notebook)
try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    # fallback: gõ tay nếu chưa setup Secrets - nhớ xoá/regenerate key sau khi test xong
    ROBOFLOW_API_KEY = "n3X74kkklU9Dskz6coWW"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("kien2k11").project("plate_character_recognition-njki9")
version = project.version(2)
dataset = version.download("yolov8")

print("📂 Dataset tải về tại:", dataset.location)


## Kiểm tra dataset trước khi train

⭐ Kiểm tra lại `data.yaml` xem danh sách class (`names`) có đủ 0-9 + các chữ cái cần dùng
(A, H, K, M...) không, và xem thử vài ảnh + box mẫu để chắc là gán nhãn đúng trước khi tốn thời
gian train.

In [ ]:
import yaml

data_yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print("Số lớp:", data_cfg.get('nc'))
print("Danh sách lớp:", data_cfg.get('names'))


In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob

# Xem thử vài ảnh train kèm box đã gán nhãn, để chắc dataset đúng trước khi train
train_img_dir = os.path.join(dataset.location, 'train', 'images')
train_lbl_dir = os.path.join(dataset.location, 'train', 'labels')
sample_imgs = sorted(glob.glob(os.path.join(train_img_dir, '*')))[:6]

names = data_cfg.get('names')

def parse_label_line(parts, w, h):
    """Trả về (cls, x1, y1, x2, y2) dù dòng là bbox (5 số) hay polygon (>5 số)."""
    cls = int(float(parts[0]))
    vals = list(map(float, parts[1:]))
    if len(vals) == 4:
        xc, yc, bw, bh = vals
        x1 = (xc - bw / 2) * w; y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w; y2 = (yc + bh / 2) * h
    else:
        # ⭐ FIX: nhãn dạng polygon (nhiều cặp x,y) chứ không phải bbox 4 số ->
        #        lấy bounding box bao quanh polygon thay vì unpack cứng 5 giá trị
        xs = [vals[i] * w for i in range(0, len(vals), 2)]
        ys = [vals[i] * h for i in range(1, len(vals), 2)]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    return cls, int(x1), int(y1), int(x2), int(y2)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.split()
                if not parts:
                    continue
                cls, x1, y1, x2, y2 = parse_label_line(parts, w, h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 1)
                label = names[cls] if cls < len(names) else str(cls)
                cv2.putText(img, label, (x1, max(0, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Train YOLOv8 để detect + đọc từng ký tự

⭐ Model nhỏ (`yolov8n`) là đủ vì object nhỏ, đơn giản (chỉ là chữ/số), train nhanh. Tăng `epochs`
nếu dataset còn ít ảnh và model chưa hội tụ tốt.

In [ ]:
from ultralytics import YOLO

char_model = YOLO('yolov8n.pt')

results = char_model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=320,          # ⭐ ảnh ký tự/biển nhỏ, không cần imgsz lớn như detect biển trong cả khung hình
    batch=16,
    patience=20,        # ⭐ early stop nếu không cải thiện sau 20 epoch
    project='/kaggle/working/yolo_char_runs',
    name='train',
)

BEST_CHAR_WEIGHTS = os.path.join(results.save_dir, 'weights', 'best.pt')
print("✅ Train xong, weights tốt nhất tại:", BEST_CHAR_WEIGHTS)


## Test nhận diện: gộp các box thành 1 chuỗi biển số

YOLO trả về các box KHÔNG theo thứ tự đọc — cần tự sắp xếp lại theo đúng thứ tự đọc trái→phải,
trên→dưới (biển 2 hàng) trước khi ghép thành chuỗi.

In [ ]:
import numpy as np

def boxes_to_plate_string(result, row_gap_ratio=0.5):
    """Sắp xếp các box ký tự YOLO trả về theo đúng thứ tự đọc rồi ghép thành chuỗi."""
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return ""

    names = result.names
    items = []
    for b in boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        cls = int(b.cls[0])
        conf = float(b.conf[0])
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        items.append({"cx": cx, "cy": cy, "h": y2 - y1, "label": names[cls], "conf": conf})

    if not items:
        return ""

    avg_h = np.mean([it["h"] for it in items])
    items.sort(key=lambda it: it["cy"])

    # ⭐ gom thành các hàng dựa theo khoảng cách theo trục y (biển 1 hàng sẽ chỉ ra 1 hàng)
    rows = []
    current_row = [items[0]]
    for it in items[1:]:
        if abs(it["cy"] - current_row[-1]["cy"]) > avg_h * row_gap_ratio:
            rows.append(current_row)
            current_row = [it]
        else:
            current_row.append(it)
    rows.append(current_row)

    plate_str = ""
    for row in rows:
        row.sort(key=lambda it: it["cx"])
        plate_str += "".join(it["label"] for it in row)
    return plate_str


In [ ]:
char_model_trained = YOLO(BEST_CHAR_WEIGHTS)

# ⭐ đổi đường dẫn này thành 1 ảnh biển số thật để test thử (crop sẵn, hoặc cả ảnh gốc đều được)
TEST_IMG = sample_imgs[0]

result = char_model_trained(TEST_IMG, conf=0.4, verbose=False)[0]
plate_str = boxes_to_plate_string(result)
print("📋 Đọc được:", plate_str)

annotated = result.plot()
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()
